# Lab 13: Optimize AI Agents with Fine-Tuning

> **Source:** Microsoft Learning -- [mslearn-genaiops](https://github.com/MicrosoftLearning/mslearn-genaiops), `docs/06-optimize-finetuning.md`
> **License:** MIT

## Objectives

By the end of this lab you will:

1. Understand **when fine-tuning is needed** (and when it is not)
2. Compare three fine-tuning methods: **SFT**, **RFT**, **DPO**
3. Apply a **decision framework** to choose the right method
4. Understand how fine-tuning fits into the GenAIOps lifecycle

| Estimated Time | Estimated Cost |
|---|---|
| ~15 minutes | Free (conceptual lab -- no Azure resources needed) |

## When Is Fine-Tuning Needed?

Fine-tuning is a **last resort**, not a first step. Consider it only after evaluation scores **plateau** despite prompt optimization.

### Stay with prompt engineering if:
- Evaluation scores are **still improving** with prompt changes (Labs 09--10)
- The task is **well-covered** by the base model's training data
- You need **rapid iteration** (fine-tuning takes hours/days; prompt changes take minutes)
- You have **limited budget or data** (fine-tuning requires compute and labeled examples)

### Consider fine-tuning if:
- Evaluation scores have **plateaued** across multiple prompt experiments
- The model consistently fails on **domain-specific** tasks (e.g., medical terminology, legal reasoning)
- You need a **specific output format** the model cannot reliably produce via prompting alone
- You need to **reduce token usage** by baking instructions into model weights

> **Exam Tip:** Fine-tuning is a **last resort** in the GenAIOps lifecycle. The expected progression is: prompt engineering -> prompt optimization -> evaluation -> fine-tuning (only if needed). The exam may present scenarios where fine-tuning is premature.

## Method 1: Supervised Fine-Tuning (SFT)

**What:** Train the model on labeled (input, output) pairs. You show the model exactly what good responses look like.

**Data format:** JSONL with `messages` array:

```json
{"messages": [{"role": "system", "content": "You are a professional trail guide."}, {"role": "user", "content": "What gear for a day hike?"}, {"role": "assistant", "content": "Essential day hike gear: ..."}]}
{"messages": [{"role": "system", "content": "You are a professional trail guide."}, {"role": "user", "content": "Bear safety tips?"}, {"role": "assistant", "content": "Bear encounter protocol: ..."}]}
```

**Best for:**
- **Domain adaptation** -- teaching the model specialized vocabulary or knowledge
- **Consistent output formatting** -- enforcing a specific response structure
- Tasks where you have **hundreds to thousands** of labeled examples

**Risks:**
- **Catastrophic forgetting** -- the model may lose general capabilities it had before fine-tuning
- Requires **high-quality labeled data** -- garbage in, garbage out
- **Slower iteration** -- each training run takes hours

## Method 2: Reinforcement Fine-Tuning (RFT)

**What:** A reward function or reward model scores the model's outputs, and the model is optimized to maximize that reward.

**Key insight:** You can use the **cloud evaluators from Lab 11** (IntentResolution, Relevance, Groundedness) as reward signals. The evaluation pipeline you already built becomes your training signal.

**Best for:**
- **Metric optimization** -- directly optimize for the scores you care about
- **Reasoning improvement** -- reward correct reasoning chains, not just final answers
- Tasks where you have **reliable automated evaluators** but fewer labeled examples than SFT needs

**Trade-offs:**
- Fewer examples needed than SFT
- Harder to control -- reward hacking is possible (model finds shortcuts to maximize score without genuine improvement)
- Requires a **reliable reward signal** -- if your evaluator is noisy, training will be noisy

## Method 3: Direct Preference Optimization (DPO)

**What:** Train on **preference pairs** -- for each prompt, provide a "chosen" (good) and "rejected" (bad) response. The model learns to prefer the chosen response without needing a separate reward model.

**Data format:**

```json
{"prompt": "What gear for a day hike?", "chosen": "Essential gear: water (2L), snacks, sun protection, first aid kit, navigation (map/phone). Wear moisture-wicking layers.", "rejected": "You'll need lots of stuff. Bring everything you can think of. Maybe a tent? Some people bring hammocks too. Don't forget your favorite snacks and a good book to read."}
```

**Best for:**
- **Style and tone control** -- teaching the model to prefer concise over verbose, formal over casual
- **Safety alignment** -- teaching the model to refuse unsafe requests (chosen = refusal, rejected = unsafe response)
- Tasks where you have **comparative judgments** ("A is better than B") but not absolute labels

**Trade-offs:**
- Simpler than RFT (no separate reward model)
- Requires **paired examples** -- harder to collect than single labeled examples
- Most effective for **stylistic** rather than factual improvements

## Decision Framework

Use this table to choose the right fine-tuning method -- or to decide that fine-tuning is not needed yet:

| You have... | Use | Example |
|---|---|---|
| Labeled (input, output) examples | **SFT** | 500 trail-guide Q&A pairs from expert rangers |
| Reliable automated evaluators | **RFT** | Lab 11 cloud evaluators as reward signal |
| Comparative judgments (A > B) | **DPO** | "This concise answer is better than that verbose one" |
| None of the above | **Stay with prompt engineering** | Continue Labs 09--10 optimization loop |

### Decision Flowchart

```
Have evaluation scores plateaued despite prompt optimization?
  NO  --> Stay with prompt engineering (Labs 09-10)
  YES --> Do you have labeled examples?
            YES --> SFT
            NO  --> Do you have reliable evaluators?
                      YES --> RFT
                      NO  --> Do you have preference pairs?
                                YES --> DPO
                                NO  --> Collect data first
```

## Fine-Tuning in the GenAIOps Lifecycle

Fine-tuning **extends** the workflow from Labs 08--12 -- it does **not replace** it.

A fine-tuned model still needs:
- **Versioning** (Lab 09) -- tag the fine-tuned model version in Git alongside the training data and hyperparameters
- **Evaluation** (Lab 11) -- run the same 89-item evaluation against the fine-tuned model to verify improvement
- **Monitoring** (Lab 12) -- instrument the fine-tuned model with OpenTelemetry to track production performance

The full lifecycle becomes:

```
Prompt Engineering (Labs 09-10)
  --> Automated Evaluation (Lab 11)
    --> Monitoring (Lab 12)
      --> Scores plateau?
        --> Fine-Tune (Lab 13)
          --> Re-evaluate (Lab 11 again)
            --> Re-monitor (Lab 12 again)
```

Fine-tuning adds a model training step to the loop, but the evaluation and monitoring infrastructure is reused.

## Key Takeaways

| Concept | Detail |
|---|---|
| **Fine-tuning is a last resort** | Only after evaluation scores plateau despite prompt optimization. The exam expects you to exhaust prompt engineering first. |
| **SFT risks catastrophic forgetting** | The model may lose general capabilities. Always evaluate on a broad test set, not just the fine-tuning domain. |
| **RFT leverages existing evaluators** | Lab 11's cloud evaluators can serve as reward signals -- reuse infrastructure you already built. |
| **DPO uses preference pairs** | Simpler than RFT (no reward model), best for style/tone/safety alignment. |
| **Fine-tuned models need full lifecycle** | Versioning, evaluation, monitoring -- everything from Labs 08--12 still applies. Fine-tuning adds a step; it does not skip steps. |